# T-test and ANOVA

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import scipy.stats as stats

import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
df = pd.read_csv("ai_jobs_market_2025_2026.csv")
df.head(5)

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,...,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7,Master's,239000.0,155000,290000,Boston,...,13.1,96,16.9,6.8,2026,3,1,0,1,Senior ($200-300k)
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2,Bachelor's,166000.0,90000,200000,London,...,5.4,82,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k)
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4,Associate's,360000.0,160000,300000,Seattle,...,9.1,98,42.7,7.7,2026,1,1,1,1,Elite (>$300k)
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3,Bachelor's,161000.0,130000,220000,Singapore,...,12.0,88,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k)
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,...,9.4,85,17.3,8.9,2026,1,1,1,0,Senior ($200-300k)


In [3]:
df.columns

Index(['job_id', 'job_title', 'job_category', 'experience_level',
       'years_of_experience', 'education_required', 'annual_salary_usd',
       'salary_min_usd', 'salary_max_usd', 'city', 'country', 'remote_work',
       'company_size', 'industry', 'required_skills', 'ai_salary_premium_pct',
       'demand_score', 'demand_growth_yoy_pct', 'benefits_score_10',
       'posting_year', 'posting_month', 'is_senior', 'is_remote_friendly',
       'is_llm_role', 'salary_tier'],
      dtype='str')

In [4]:
remote = df[df["remote_work"] == "Fully Remote"]["annual_salary_usd"].dropna()
in_person = df[df["remote_work"] == "On-site"]["annual_salary_usd"].dropna()

In [5]:
t_stat, p_value = stats.ttest_ind(remote, in_person)
print("t-statistic:", t_stat)
print("p-value:", p_value)

t-statistic: 0.5587308399484088
p-value: 0.5764994720992597


The Two Sample t-test was conudcted to determine whether there was a significat difference between annual salries in remote vs in person groups. The t-stat is $0.5589$ and the p-value is $0.5764$,. The p-value is greater than the signifcance level of $0.05$, so we fail to reject the null hypothesis, meaning that there is insufficient evidence to suggest that average salaries differ significantly between two remote work groups. 

In [6]:
small_formula = 'annual_salary_usd ~ years_of_experience'
big_formula = 'annual_salary_usd ~ years_of_experience + C(job_category)'
small_model = smf.ols(small_formula, data=df).fit()
big_model = smf.ols(big_formula, data=df).fit()
anova_table = sm.stats.anova_lm(small_model, big_model)
print(anova_table)

   df_resid           ssr  df_diff       ss_diff          F        Pr(>F)
0    1498.0  6.589307e+12      0.0           NaN        NaN           NaN
1    1487.0  5.770775e+12     11.0  8.185314e+11  19.174297  2.098907e-36


In [7]:
f_stat = anova_table['F'].values[1]
p_value = anova_table['Pr(>F)'].values[1]
print("f-statistic:", f_stat)
print("p-value:", p_value)

f-statistic: 19.174297148451828
p-value: 2.0989071901160962e-36


The one-way ANOVA resulted in a f-stat of $19.174$ and a p-value of $2.098$. The p-value is below the significance level, which tells us to reject null hypothesis that all jobs have the same mean salary. 

In [15]:
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols

df['log_salary'] = np.log(df['annual_salary_usd'])

formula_city_size = 'log_salary ~ C(city) * C(company_size)'
model_city_size = ols(formula_city_size, data=df).fit()

anova_city_size = sm.stats.anova_lm(model_city_size, typ=2)
display(anova_city_size)

,sum_sq,df,F,PR(>F)
C(city),33.158309,19.0,22.874064,4.385792e-69
C(company_size),16.774011,4.0,54.964398,4.346602e-43
C(city):C(company_size),8.091595,76.0,1.395485,1.569023e-02
Residual,106.812845,1400.0,NaN,NaN


The Two-Way ANOVA results show statistically significant main effects for both City (p=4.39×10 
−69
 ) and Company Size (p=4.35×10 
−43
 ), indicating both independently impact log salary. Furthermore, the significant interaction effect (p=0.0157) confirms that the effect of company size on salary varies depending on the city location, justifying the inclusion of this interaction term in our final regression model.

In [ ]:
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols

df['log_salary'] = np.log(df['annual_salary_usd'])

model_edu = ols('log_salary ~ C(education_required)', data=df).fit()
anova_edu = sm.stats.anova_lm(model_edu, typ=2)
display(anova_edu)

model_city = ols('log_salary ~ C(city)', data=df).fit()
anova_city = sm.stats.anova_lm(model_city, typ=2)
display(anova_city)

,sum_sq,df,F,PR(>F)
C(education_required),6.219985,4.0,14.366395,1.632214e-11
Residual,161.816459,1495.0,NaN,NaN


,sum_sq,df,F,PR(>F)
C(city),36.357994,19.0,21.507668,2.214440e-65
Residual,131.678450,1480.0,NaN,NaN


The One-Way ANOVA tests reveal that both education required (p=1.63×10 
−11
 ) and city location (p=2.21×10 
−65
 ) have highly statistically significant main effects on log salary. These exceptionally small p-values indicate that mean salary levels vary significantly across different education categories and geographic locations, making both variables excellent standalone predictors for our analysis.